In [26]:
import pandas as pd
import plotly.express as px 
import plotly.graph_objects as go
import numpy as np

In [27]:
path = "https://fullstackds-projects-bucket.s3.eu-west-3.amazonaws.com/data/Getaround/get_around_delay_analysis.xlsx"
df = pd.read_excel(path)
df.head()

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
0,505000,363965,mobile,canceled,NaN,NaN,NaN
1,507750,269550,mobile,ended,-81.0,NaN,NaN
2,508131,359049,connect,ended,70.0,NaN,NaN
3,508865,299063,connect,canceled,NaN,NaN,NaN
4,511440,313932,mobile,ended,NaN,NaN,NaN


In [34]:
Q1 = df["delay_at_checkout_in_minutes"].quantile(0.25)
Q3 = df["delay_at_checkout_in_minutes"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
print(lower)
print(upper)

-190.5
221.5


In [29]:
# Add outliers flag
df["is_outlier"] = (
    (df["delay_at_checkout_in_minutes"] < lower)
    | (df["delay_at_checkout_in_minutes"] > upper)
)

In [30]:
df.head()


,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes,is_outlier
0,505000,363965,mobile,canceled,NaN,NaN,NaN,False
1,507750,269550,mobile,ended,-81.0,NaN,NaN,False
2,508131,359049,connect,ended,70.0,NaN,NaN,False
3,508865,299063,connect,canceled,NaN,NaN,NaN,False
4,511440,313932,mobile,ended,NaN,NaN,NaN,False


In [31]:
df["time_delta_with_previous_rental_in_minutes"].dtype

dtype('float64')

In [32]:
rentals_clean = df[
    [
        "rental_id",
        "car_id",
        "checkin_type",
        "state",
        "delay_at_checkout_in_minutes",
        "time_delta_with_previous_rental_in_minutes",
        "is_outlier"
    ]
].copy()

rentals_clean.to_csv(
    "rentals_clean.csv",
    index=False
)

In [33]:
# Create buffers statistics table
import pandas as pd

# Thresholds to test (minutes)
thresholds = [30, 60, 90, 120, 150, 180, 240]

# Keep only rentals with necessary information
df_check = df[
    df["delay_at_checkout_in_minutes"].notna() &
    df["time_delta_with_previous_rental_in_minutes"].notna()
].copy()

# Real conflict cases:
# delay exceeds available gap before next rental
conflict_cases = df_check[
    df_check["delay_at_checkout_in_minutes"] >
    df_check["time_delta_with_previous_rental_in_minutes"]
].copy()

results = []

for t in thresholds:

    # Rentals that would be impacted by introducing a buffer
    affected_rentals = df_check[
        df_check["delay_at_checkout_in_minutes"] >
        (
            df_check["time_delta_with_previous_rental_in_minutes"]
            - t
        )
    ].shape[0]

    # Existing conflicts that would be prevented
    saved_rentals = conflict_cases[
        conflict_cases["time_delta_with_previous_rental_in_minutes"] < t
    ].shape[0]

    efficiency = (
        saved_rentals / affected_rentals
        if affected_rentals > 0
        else 0
    )

    results.append({
        "buffer_minutes": t,
        "saved_rentals": saved_rentals,
        "affected_rentals": affected_rentals,
        "efficiency": round(efficiency, 3)
    })



buffer_analysis = pd.DataFrame(results)

buffer_analysis["efficiency_pct"] = (
    buffer_analysis["efficiency"] * 100
).round(1)

buffer_analysis = buffer_analysis.drop(columns=["efficiency"])

print(buffer_analysis)

# Export for Power BI
buffer_analysis.to_csv(
    "buffer_analysis.csv",
    index=False
)

   buffer_minutes  saved_rentals  affected_rentals  efficiency_pct
0              30            136               340            40.0
1              60            176               426            41.3
2              90            213               516            41.3
3             120            224               597            37.5
4             150            241               671            35.9
5             180            245               736            33.3
6             240            251               829            30.3
